# 3. Parsing statistics

*Author: La Anh Hào · code: `src/13_dedup_export.ipynb`, `src/15_run_parsing.ipynb`*

This chapter reports what the parser produced from the corpus of Milestone 1: how many versions it could
parse, how much of each paper's history is covered, and how many elements and BibTeX entries remain after the
two deduplication steps. Every number and interactive chart is computed when the book is built from the log
files written by `src/15_run_parsing.ipynb` (`m2_stats.json`, `m2_parse_status.csv`, and `m2_papers.csv`).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import report_data as rd

rd.check()
stats = rd.stats()
versions, papers = rd.versions(), rd.papers()
status = pd.Series({k: int(stats["status"].get(k, 0)) for k in rd.STATUSES})
with_tex = int(stats["versions_with_tex"])
rd.md(f"Parser version `{stats['parser_version']}`; statistics computed on "
      f"{stats['computed_at'][:16].replace('T', ' ')} UTC over **{stats['papers']:,}** papers and "
      f"**{stats['versions']:,}** versions.")

## 3.1 Parse success rate

**Definition.** The statement asks for every version of every paper, including the versions without LaTeX,
which stay in the corpus as empty records. Each version listed in the paper's `metadata.json` ends with one
status:

- `ok`: a root file was found, the document tree was built and its rows were written;
- `empty_source`: the version has no `.tex` file (a PDF-only submission, a persistent HTTP 404 at `/src/`,
  or an unextractable archive from Milestone 1); it is recorded, with no rows;
- `no_root`: the version has `.tex` files but none of them is a LaTeX document (`\documentclass`);
- `error`: an exception in one of the steps; the message is logged and the other versions go on.

The **parse success rate** is the share of `ok` among the versions that have LaTeX source (`ok`, `no_root`
and `error`). The share among all versions is given too; the difference is the versions without LaTeX source,
which no parser can read.

In [ ]:
labels = {"ok": "`ok`", "empty_source": "`empty_source`", "no_root": "`no_root`", "error": "`error`"}
t31 = pd.DataFrame({
    "Status": [labels[k] for k in rd.STATUSES],
    "Versions": status.values,
    "Share of all versions": 100 * status.values / stats["versions"],
    "Share of versions with LaTeX": [np.nan if k == "empty_source" else 100 * v / with_tex for k, v in status.items()],
})
rd.table(t31, "Table 3.1 – Status of every version listed in metadata.json",
         {"Share of all versions": "{:.2f}%", "Share of versions with LaTeX": "{:.2f}%"})
rd.md(f"**Parse success rate: {stats['parse_success_rate_of_tex_versions_pct']:.2f}%** of the {with_tex:,} versions "
      f"with LaTeX source ({status['ok']:,} versions), and {stats['parse_success_rate_of_all_versions_pct']:.2f}% of "
      f"all {stats['versions']:,} versions. "
      + ("No version failed with an exception." if status["error"] == 0
         else f"{status['error']:,} versions failed with an exception (listed in `m2_parse_status.csv`)."))

In [ ]:
tex = versions["status"] != "empty_source"
t32 = (versions.assign(tex=tex, ok=versions["status"] == "ok", no_root=versions["status"] == "no_root",
                       error=versions["status"] == "error")
       .groupby("range")
       .agg(papers=("arxiv_id", "nunique"), versions=("status", "size"), tex=("tex", "sum"), ok=("ok", "sum"),
            no_root=("no_root", "sum"), error=("error", "sum")))
t32["rate"] = 100 * t32["ok"] / t32["tex"]
t32.loc["all"] = t32.sum(numeric_only=True)
t32.loc["all", "rate"] = 100 * t32.loc["all", "ok"] / t32.loc["all", "tex"]
t32 = t32.reset_index().rename(columns={
    "range": "Range", "papers": "Papers", "versions": "Versions", "tex": "With LaTeX", "ok": "ok",
    "no_root": "no_root", "error": "error", "rate": "Success rate"})
rd.table(t32, "Table 3.2 – Parse success rate per ID range (R1–R4 as in chapter 1 of Milestone 1)",
         {"Success rate": "{:.2f}%"})

In [ ]:
order = ["ok", "no_root", "error", "empty_source"]
colors = {"ok": rd.BLUE, "no_root": rd.ORANGE, "error": rd.MAGENTA, "empty_source": rd.AXIS}
counts = versions.groupby(["range", "status"]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
share = 100 * counts.div(counts.sum(axis=1), axis=0)

fig = go.Figure()
for name in order:
    fig.add_trace(go.Bar(
        y=share.index.tolist(),
        x=share[name].tolist(),
        name=name,
        orientation="h",
        marker_color=colors[name],
        customdata=counts[name].to_numpy()[:, None],
        hovertemplate="%{y} · <b>" + name + "</b><br>%{customdata[0]:,} versions (%{x:.2f}%)<extra></extra>",
    ))
fig.update_layout(
    barmode="stack",
    title="Status of the versions per ID range",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(title_text="Share of the versions of the range (%)", range=[0, 100])
rd.show(fig, height=320, margin=dict(l=45, r=25, t=55, b=75))
rd.figure_caption("Figure 3.1 – Status of the versions of each range. Hover over any bar for exact counts; "
                  "the grey part is the versions without LaTeX source.")

### Versions without a root

The statement defines the root as the `.tex` file that holds `\documentclass`. A version whose files never
declare a document class has no root and is not parsed. Checked one by one from parser 13.8 onwards
(unchanged in 13.9), all seven are **Plain TeX** papers (`\magnification`, `\input` of macro files, `\bye`
or `\end` at the end), which never were LaTeX documents. Before 13.8, two more versions were in this list
because their files end lines with a carriage return only (an old Mac format): read as one single line, a `%`
on the first line turned the whole file into a comment (section [2.1](hierarchy.md)).

In [ ]:
no_root = versions.loc[versions["status"] == "no_root", ["arxiv_id", "version", "error"]]
if len(no_root):
    rd.table(no_root.rename(columns={"arxiv_id": "Paper", "version": "Version", "error": "Logged reason"}),
             f"Table 3.3 – The {len(no_root)} versions with LaTeX files but no root")
else:
    rd.md("Every version with LaTeX files has a root.")

### Versions without LaTeX source

These versions are part of the corpus as empty records (no rows in `elements` or `hierarchy`). The reason is
taken from the download receipts and error logs of Milestone 1 ([3.1](../milestone1/statistics.ipynb)).

In [ ]:
reason_labels = {
    "pdf_only": "arXiv has no LaTeX source for the version (PDF only)",
    "files_missing": "a download receipt exists but the vN/ folder is not on Drive",
    "not_found": "source request returned persistent HTTP 404 (kept as empty record)",
    "not_downloaded": "archive failed extraction/safety checks in Milestone 1 (no receipt)",
    "": "no reason logged",
}
reasons = pd.Series(stats.get("empty_source_reasons") or {}, dtype="int64").sort_values(ascending=False)
t34 = pd.DataFrame({"Reason": [reason_labels.get(k, k) for k in reasons.index], "Versions": reasons.values,
                    "Share of all versions": 100 * reasons.values / stats["versions"]})
rd.table(t34, f"Table 3.4 – Why {status['empty_source']:,} versions have no LaTeX source",
         {"Share of all versions": "{:.2f}%"})
rd.md(
    f"The counts reconcile exactly with [Milestone 1 Table 3.2](../milestone1/statistics.ipynb): "
    f"**{int(reasons.get('pdf_only', 0)):,}** PDF-only receipts, **{int(reasons.get('not_found', 0)):,}** "
    f"persistent `/src/` HTTP 404 versions, and **{int(reasons.get('not_downloaded', 0)):,}** unextractable "
    f"archives, with **0** `files_missing` versions after `09_repair_missing_sources.ipynb` ran.")

## 3.2 Version coverage of every paper

**Definition.** The *version coverage* of a paper is the number of its versions that were parsed over the
number of versions it has on arXiv (`metadata.json`). A paper is *fully covered* when every version was
parsed, *partly covered* when some were, and *not covered* when none was, which is the case of a paper whose
versions are all PDF-only.

In [ ]:
dist = stats["version_coverage"]["distribution"]
cov = pd.DataFrame([(int(key.split("/")[1]), int(key.split("/")[0]), n) for key, n in dist.items()],
                   columns=["versions", "parsed", "papers"])
cov["group"] = np.select([cov["parsed"] == cov["versions"], cov["parsed"] == 0], ["full", "none"], "partial")
cov["n"] = cov["versions"].clip(upper=5)
by_n = cov.pivot_table(index="n", columns="group", values="papers", aggfunc="sum", fill_value=0)
by_n = by_n.reindex(columns=["full", "partial", "none"], fill_value=0)
names = {n: (f"{n}" if n < 5 else "5 or more") for n in by_n.index}
share = 100 * by_n.div(by_n.sum(axis=1), axis=0)
xs = [names[n] for n in share.index]
totals_n = by_n.sum(axis=1)

fig = go.Figure()
for group, color in (("full", rd.BLUE), ("partial", rd.YELLOW), ("none", rd.AXIS)):
    fig.add_trace(go.Bar(
        x=xs, y=share[group].tolist(), name=group, marker_color=color,
        customdata=np.stack([by_n[group].to_numpy(), totals_n.to_numpy()], axis=-1),
        hovertemplate="%{x} version(s) · <b>" + group + "</b><br>%{customdata[0]:,} of %{customdata[1]:,} papers "
                      "(%{y:.2f}%)<extra></extra>",
    ))
for x_lbl, tot in zip(xs, totals_n):
    fig.add_annotation(x=x_lbl, y=102, text=f"{tot:,}", showarrow=False, font=dict(size=10, color="#52514e"))
fig.update_layout(
    barmode="stack",
    title="Version coverage per paper",
    legend=dict(title="Versions parsed", orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Versions of the paper on arXiv", showgrid=False)
fig.update_yaxes(title_text="Share of the papers (%)", range=[0, 112])
rd.show(fig, height=360, margin=dict(l=55, r=25, t=55, b=80))
rd.figure_caption("Figure 3.2 – For the papers with 1, 2, ... versions: share whose versions were all parsed "
                  "(full), some (partial) or none. The number of papers is written above each bar.")

In [ ]:
t35 = by_n.copy()
t35["papers"] = t35.sum(axis=1)
t35["full_pct"] = 100 * t35["full"] / t35["papers"]
t35 = t35.reset_index()
t35["n"] = t35["n"].map(names)
t35 = t35.rename(columns={"n": "Versions on arXiv", "papers": "Papers", "full": "Every version parsed",
                          "partial": "Some versions", "none": "No version", "full_pct": "Fully covered"})
t35 = t35[["Versions on arXiv", "Papers", "Every version parsed", "Some versions", "No version", "Fully covered"]]
rd.table(t35, "Table 3.5 – Version coverage by number of versions", {"Fully covered": "{:.2f}%"})
c = stats["version_coverage"]
rd.md(f"**{c['papers_full']:,}** papers ({rd.pct(c['papers_full'], stats['papers'])}) have every version parsed, "
      f"**{c['papers_partial']:,}** some of them and **{c['papers_none']:,}** none. Over all papers, a paper has "
      f"**{c['mean_pct']:.2f}%** of its versions parsed on average.")

## 3.3 Size of the parsed documents

The size of a version is the number of nodes of its tree, from the `document` root to the sentences,
formulas and figures (chapter [2.1](hierarchy.md)). Use the dropdown in Figure 3.3 to compare the four ID
ranges.

In [ ]:
ok = versions[versions["status"] == "ok"]
nodes = ok["n_nodes"].clip(lower=1)
bins = np.logspace(0, np.log10(nodes.max()) + 0.05, 60)
centers = np.sqrt(bins[:-1] * bins[1:])

subsets = [("All ranges", ok)] + [(r, ok[ok["range"] == r]) for r in rd.RANGES]
fig = go.Figure()
buttons = []
for idx, (lbl, sub) in enumerate(subsets):
    vals = sub["n_nodes"].clip(lower=1)
    h, edges = np.histogram(vals, bins=bins)
    fig.add_trace(go.Bar(
        x=centers, y=h, marker_color=rd.BLUE, visible=(idx == 0), showlegend=False,
        customdata=np.stack([edges[:-1], edges[1:]], axis=-1),
        hovertemplate=f"{lbl}<br>%{{customdata[0]:,.0f}}–%{{customdata[1]:,.0f}} nodes<br>"
                      "<b>%{y:,}</b> versions<extra></extra>",
    ))
    vis = [i == idx for i in range(len(subsets))]
    buttons.append(dict(label=f"{lbl} (median {vals.median():,.0f})", method="update", args=[{"visible": vis}]))

fig.add_vline(
    x=float(nodes.median()), line_color=rd.ORANGE, line_width=1.8, line_dash="dash",
    annotation_text=f"overall median {nodes.median():,.0f}", annotation_position="top right",
)
fig.update_layout(
    title="Size of the tree of each parsed version",
    updatemenus=[dict(active=0, buttons=buttons, x=1.0, xanchor="right", y=1.18, yanchor="top")],
)
fig.update_xaxes(type="log", title_text="Nodes in the tree of one version (log scale)")
fig.update_yaxes(title_text="Versions")
rd.show(fig, height=360, margin=dict(l=55, r=25, t=75, b=45))
rd.figure_caption("Figure 3.3 – Number of tree nodes per parsed version (dropdown filters by ID range).")
q = nodes.quantile([0.1, 0.5, 0.9, 0.99])
rd.md(f"A parsed version has **{nodes.median():,.0f}** nodes at the median (10% of the versions have fewer "
      f"than {q[0.1]:,.0f}, 10% more than {q[0.9]:,.0f}; the largest has {nodes.max():,}). "
      f"{int((ok['n_nodes'] <= 1).sum()):,} parsed versions have a tree with the document node only.")

## 3.4 Elements before and after deduplication

**Definitions.** *Before deduplication* counts every node of every version's tree. *After deduplication*
counts the rows of the `elements` table, one per distinct element of a paper over all of its versions
(chapter [2.4](deduplication.ipynb)): a sentence, formula or figure whose cleaned content is identical in two
versions, or twice in one version, is one element with several `hierarchy` rows. A section is shared when
it has the same title under the same parent.

In [ ]:
before = pd.Series(stats["elements_before_dedup_by_kind"], dtype="int64")
after = pd.Series(stats["elements_after_dedup_by_kind"], dtype="int64")
kinds = [k for k in rd.KINDS if k in before.index or k in after.index]
kinds += [k for k in before.index.union(after.index) if k not in kinds]
t36 = pd.DataFrame({"Kind": [f"`{k}`" for k in kinds],
                    "Before (nodes)": before.reindex(kinds).fillna(0).astype(int).values,
                    "After (elements)": after.reindex(kinds).fillna(0).astype(int).values})
t36["Removed"] = 100 * (1 - t36["After (elements)"] / t36["Before (nodes)"].where(t36["Before (nodes)"] > 0))
total = pd.DataFrame({"Kind": ["**all kinds**"], "Before (nodes)": [stats["elements_before_dedup"]],
                      "After (elements)": [stats["elements_after_dedup"]],
                      "Removed": [100 * (1 - stats["elements_after_dedup"] / stats["elements_before_dedup"])]})
rd.table(pd.concat([t36, total], ignore_index=True), "Table 3.6 – Elements per kind before and after deduplication",
         {"Removed": "{:.1f}%"})

In [ ]:
b_vals = before.reindex(kinds).fillna(0).astype(int)
a_vals = after.reindex(kinds).fillna(0).astype(int)
rem_pct = 100 * (1 - a_vals / b_vals.where(b_vals > 0))

fig = go.Figure()
fig.add_trace(go.Bar(
    y=kinds, x=b_vals.clip(lower=1).tolist(), name="before: nodes of every version",
    orientation="h", marker_color=rd.BLUE,
    customdata=np.stack([b_vals.to_numpy(), rem_pct.to_numpy()], axis=-1),
    hovertemplate="<b>%{y}</b> (before)<br>%{customdata[0]:,.0f} nodes (dedup removes %{customdata[1]:.1f}%)<extra></extra>",
))
fig.add_trace(go.Bar(
    y=kinds, x=a_vals.clip(lower=1).tolist(), name="after: rows of elements",
    orientation="h", marker_color=rd.ORANGE,
    customdata=np.stack([a_vals.to_numpy(), rem_pct.to_numpy()], axis=-1),
    hovertemplate="<b>%{y}</b> (after)<br>%{customdata[0]:,.0f} elements (%{customdata[1]:.1f}% fewer)<extra></extra>",
))
fig.update_layout(
    barmode="group",
    title="Elements per kind, before and after deduplication",
    legend=dict(orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5),
)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(type="log", title_text="Count (log scale)")
rd.show(fig, height=460, margin=dict(l=110, r=25, t=55, b=75))
rd.figure_caption("Figure 3.4 – The same numbers as Table 3.6, on a log scale.")

The reduction depends on how many versions a paper has. With one version, only the repetitions inside the
paper are merged (a formula written twice, a sentence repeated in a caption and in the text, the same
untitled list); with two or more, every sentence, formula and figure that did not change between versions is
stored once.

In [ ]:
parsed = papers[papers["versions_ok"] > 0].copy()
parsed["n"] = parsed["versions_ok"].clip(upper=4)
t37 = parsed.groupby("n").agg(papers=("arxiv_id", "size"), nodes=("nodes", "sum"), elements=("elements", "sum"),
                              bib_before=("bib_before", "sum"), bib_after=("bib_after", "sum"))
t37["el_removed"] = 100 * (1 - t37["elements"] / t37["nodes"])
t37["bib_removed"] = 100 * (1 - t37["bib_after"] / t37["bib_before"].where(t37["bib_before"] > 0))
labels_n = [f"{n}" if n < 4 else "4 or more" for n in t37.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels_n, y=t37["el_removed"].tolist(), name="elements", marker_color=rd.BLUE,
    text=[f"{v:.1f}%" for v in t37["el_removed"]], textposition="outside",
    customdata=np.stack([t37["nodes"].to_numpy(), t37["elements"].to_numpy()], axis=-1),
    hovertemplate="%{x} parsed version(s) · elements<br>%{customdata[0]:,.0f} → %{customdata[1]:,.0f} "
                  "(<b>−%{y:.1f}%</b>)<extra></extra>",
))
fig.add_trace(go.Bar(
    x=labels_n, y=t37["bib_removed"].tolist(), name="BibTeX entries", marker_color=rd.ORANGE,
    text=[f"{v:.1f}%" for v in t37["bib_removed"]], textposition="outside",
    customdata=np.stack([t37["bib_before"].to_numpy(), t37["bib_after"].to_numpy()], axis=-1),
    hovertemplate="%{x} parsed version(s) · BibTeX<br>%{customdata[0]:,.0f} → %{customdata[1]:,.0f} "
                  "(<b>−%{y:.1f}%</b>)<extra></extra>",
))
fig.update_layout(
    barmode="group",
    title="Share removed by deduplication, by number of parsed versions",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Versions of the paper that were parsed", showgrid=False)
fig.update_yaxes(title_text="Removed by deduplication (%)", range=[0, max(t37[["el_removed", "bib_removed"]].max().max(), 1) * 1.25])
rd.show(fig, height=360, margin=dict(l=55, r=25, t=55, b=75))
rd.figure_caption("Figure 3.5 – Share of the nodes (blue) and of the extracted BibTeX entries (orange) that "
                  "deduplication removes, by number of parsed versions of the paper.")
t37 = t37.reset_index()
t37["n"] = labels_n
rd.table(t37.rename(columns={"n": "Parsed versions", "papers": "Papers", "nodes": "Nodes", "elements": "Elements",
                             "el_removed": "Elements removed", "bib_before": "BibTeX before",
                             "bib_after": "BibTeX after", "bib_removed": "BibTeX removed"}),
         "Table 3.7 – Deduplication by number of parsed versions",
         {"Elements removed": "{:.1f}%", "BibTeX removed": "{:.1f}%"})

## 3.5 BibTeX entries before and after deduplication

The references of a version come from one of four cases (chapter [2.3](references.md)), counted over the
versions with LaTeX source:

- `bib`: the `.bib` files named by `\bibliography{...}` or `\addbibresource{...}`;
- `bibitem`: a `thebibliography` list written in the `.tex` files, converted to BibTeX (including REVTeX
  `\bibinfo` fields since parser 13.9);
- `missing_bbl`: the paper declares a `.bib` file that its authors did not upload. arXiv compiled it into a
  `.bbl` file, which the download of Milestone 1 did not keep (only `.tex` and `.bib` were kept), so these
  references are lost;
- `none`: no bibliography declaration or `thebibliography` block (includes hand-written prose reference lists).

In [ ]:
source_labels = {"bib": "`bib`: a .bib file", "bibitem": "`bibitem`: thebibliography in the .tex",
                 "bbl": "`bbl`: a .bbl file", "missing_bbl": "`missing_bbl`: .bib declared but not shipped",
                 "none": "`none`: no bibliography"}
src = pd.Series(stats["bib_source_of_tex_versions"], dtype="int64").sort_values(ascending=False)
t38 = pd.DataFrame({"Source": [source_labels.get(k, f"`{k}`") for k in src.index], "Versions": src.values,
                    "Share": 100 * src.values / src.sum()})
rd.table(t38, "Table 3.8 – Source of the references of the versions with LaTeX", {"Share": "{:.2f}%"})
b0, b1 = stats["bib_entries_before_dedup"], stats["bib_entries_after_dedup"]
rd.md(f"The versions yield **{b0:,}** BibTeX entries, summed over versions; after deduplication inside each "
      f"paper the `bibtex` table has **{b1:,}** rows ({rd.pct(b0 - b1, b0, 1)} fewer). An entry is merged with "
      f"another when they share the key in one version, the DOI, the arXiv ID or the normalised title (with "
      f"the rules of chapter [2.3](references.md)), and the merged row keeps the union of their fields. "
      f"Table 3.7 shows that most of the reduction comes from papers with several versions, whose reference "
      f"lists are largely the same.")